# WaveSlide - Data Preparation - EDA

### Collaborators:
- Raul Andres Cerreño Zeballos
- Leicy Cristel Cahuana Lopez
- Diego Fabrizio Mucha Alvarez

## 1. Load and Clean Data


### 1.1. Import libraries

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import numpy as np

### 1.2. Define Loading and cleaning function

In [3]:
# Function for loading data
def load_data(images_root_path, annotations_root_path):
    df_rows = []

    for annotation_file in annotations_root_path.glob("*.json"):
        class_name = annotation_file.stem
        image_dir = images_root_path / class_name
        with open(annotation_file, "r") as f:
            annotations = json.load(f)
        for image_id, ann in annotations.items():
            image_path = image_dir / f"{image_id}.jpg"
            for bbox, label in zip(ann["bboxes"], ann["labels"]):
                df_rows.append({
                    # Each column of the dataframe
                    "image_id": image_id,
                    "image_path": str(image_path),
                    "image_exists": image_path.exists(),
                    "label": label,
                    "bbox_x": bbox[0],
                    "bbox_y": bbox[1],
                    "bbox_w": bbox[2],
                    "bbox_h": bbox[3],
                })

    df_annotations_raw = pd.DataFrame(df_rows)
    df_testing = df_annotations_raw[df_annotations_raw["image_exists"] == True].drop(columns=["image_exists"])
    df_testing = df_testing[~df_testing["image_id"].duplicated(keep=False)]
    df_testing = df_testing[~(df_testing["label"] == "no_gesture")]
    df_testing = df_testing[df_testing["width_height"] == "512 384"]
    
    return df_testing

## 2. Applying transformations to the images

### 2.1. Transforming to gray scale

In [ ]:
def to_grayscale(df_testing):
    # Creating the images grayscale and exporting it back to the dataset
    path = "../data/eda_1_preprocessing/images"

    Path(path).mkdir(parents=True, exist_ok=True)

    for value in df_testing["label"].unique():
        Path(f"{path}/{value}").mkdir(parents=True, exist_ok=True)

    for _, row in df_testing.iterrows():
        image_path = row["image_path"]
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        cv2.imwrite(f"{path}/{row["label"]}/{row["image_id"]}.jpg", img)